In [ ]:
import os
import torch

print("GPU VERIFICATION:")
num_gpus_total = torch.cuda.device_count() if torch.cuda.is_available() else 0
usable_gpus = []
for idx in range(num_gpus_total):
    major, minor = torch.cuda.get_device_capability(idx)
    name = torch.cuda.get_device_name(idx)
    ok = major >= 6
    if ok:
        usable_gpus.append(idx)
    print(f"  GPU {idx}: {name} (compute capability {major}.{minor}) -> {'usable' if ok else 'SKIPPED (too old for AMP)'}")

device = torch.device(f"cuda:{usable_gpus[0]}" if usable_gpus else "cpu")
if usable_gpus:
    print(f"\nSUCCESS: {len(usable_gpus)} usable GPU(s) detected. pipeline.py will use {device} as primary"
          f"{' and DataParallel across all usable GPUs during training' if len(usable_gpus) > 1 else ''}.")
else:
    print("\nWARNING: No compatible GPU detected! Pipeline will run on CPU.")
print()

repo_dir = "/kaggle/working/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}
else:
    print(f"{repo_dir} already exists -- pulling latest changes instead of re-cloning...")
    !git -C {repo_dir} pull --ff-only

os.chdir(repo_dir)
!chmod +x build_kaggle.sh
!./build_kaggle.sh

In [ ]:
import os
import shutil
import json
import torch

input_dir = "/kaggle/input/"
working_models_dir = "/kaggle/working/models"
os.makedirs(working_models_dir, exist_ok=True)

best_source_dir = None
max_iter = -1

# 1. Scan ALL directories in /kaggle/input/ to find the most recent state
if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        if "best_model.pth" in files:
            current_iter = 0
            if "pipeline_state.json" in files:
                try:
                    state_data = json.load(open(os.path.join(root, "pipeline_state.json")))
                    current_iter = state_data.get("total_iterations", 0)
                except Exception:
                    pass
            else:
                try:
                    ckpt = torch.load(os.path.join(root, "best_model.pth"), map_location="cpu", weights_only=False)
                    if isinstance(ckpt, dict):
                        current_iter = ckpt.get("iteration", 0)
                except Exception:
                    pass
            if current_iter > max_iter:
                max_iter = current_iter
                best_source_dir = root

# 2. Copy resume state (champion, buffer, logs, pipeline state) from the best found directory
if best_source_dir:
    print(f"Found most recent session data (Iteration {max_iter}) in:\n  -> {best_source_dir}")
    print("Copying to working directory to resume training...")
    for file in os.listdir(best_source_dir):
        if file.endswith((".pth", ".pt", ".csv", ".json")):
            shutil.copy(os.path.join(best_source_dir, file), working_models_dir)
    print("Resume data loaded successfully!")
else:
    print("No previous dataset attached. Starting a fresh session.")

# 3. Resolve the PINNED SENTINEL -- independent of whatever "best_model.pth" currently is.
#
# best_model.pth is EXPECTED to move forward every promotion (including forced
# ones), so it's not a safe long-term reference point on its own. sentinel_model.pth
# is deliberately different: once pinned, it should keep pointing at one specific,
# manually-verified-good checkpoint (e.g. your gen_226) for as long as you keep
# attaching this notebook's own past OUTPUT as an input dataset in future sessions.
#
# This block makes that persistence automatic:
#   - If a sentinel_model.pth already exists anywhere in /kaggle/input (i.e. it was
#     pinned in an earlier session and carried forward via that session's output),
#     reuse that exact file unchanged.
#   - Otherwise (first run, or the pin was never carried forward), pin whatever
#     champion this session resumed with as the new sentinel going forward.
#
# To deliberately MOVE the sentinel later, only do it after a proper round-robin
# via arena.py confirms a new checkpoint is robustly stronger -- then just delete
# sentinel_model.pth from the dataset you attach next session; this block will
# re-bootstrap it from that session's resumed champion.
sentinel_path = os.path.join(working_models_dir, "sentinel_model.pth")
found_existing_sentinel = None

search_roots = ([best_source_dir] if best_source_dir else []) + [input_dir]
for root_dir in search_roots:
    if not root_dir or not os.path.exists(root_dir):
        continue
    for root, dirs, files in os.walk(root_dir):
        if "sentinel_model.pth" in files:
            found_existing_sentinel = os.path.join(root, "sentinel_model.pth")
            break
    if found_existing_sentinel:
        break

if found_existing_sentinel:
    shutil.copy(found_existing_sentinel, sentinel_path)
    print(f"\nPinned sentinel found at {found_existing_sentinel} -- reusing it unchanged.")
elif os.path.exists(os.path.join(working_models_dir, "best_model.pth")):
    shutil.copy(os.path.join(working_models_dir, "best_model.pth"), sentinel_path)
    print(f"\nNo pinned sentinel found in any attached input dataset. Bootstrapping the sentinel from "
          f"this session's resumed champion (iteration {max_iter}) -> {sentinel_path}.")
    print("IMPORTANT: attach this session's OUTPUT as an input dataset next time, or this pin will be "
          "lost and silently re-bootstrapped from whatever is 'best' then -- which defeats the point.")
else:
    print("\nNo champion available yet to bootstrap a sentinel from -- skipping for now. One will be "
          "pinned automatically once a first champion exists.")

SENTINEL_CHECKPOINT = sentinel_path if os.path.exists(sentinel_path) else None
print(f"\nSENTINEL_CHECKPOINT = {SENTINEL_CHECKPOINT}")

In [ ]:
import json, os, math, torch

state_path = "/kaggle/working/models/pipeline_state.json"
model_path = "/kaggle/working/models/best_model.pth"

MAX_REJECTIONS = 5
MIN_FORCE_PROMOTE_LCB = 0.55  # must match --min-force-promote-lcb in the run cell below

total_iterations = 0
consecutive_rejections = 0
if os.path.exists(state_path):
    state_data = json.load(open(state_path))
    total_iterations = state_data.get("total_iterations", 0)
    consecutive_rejections = state_data.get("consecutive_rejections", 0)

if total_iterations == 0 and os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        total_iterations = ckpt.get("iteration", 0)
    print("(state file missing/zero -- falling back to checkpoint's embedded iteration)")

next_iter = total_iterations + 1
if next_iter <= 100:
    lr = 0.001
elif next_iter <= 250:
    lr = 0.0005
elif next_iter <= 700:
    lr = 0.0001
else:
    lr = 0.00003

print(f"total_iterations (resume point):  {total_iterations}")
print(f"next iteration will be:           {next_iter}")
print(f"learning rate that implies:       {lr}")
print(f"consecutive rejections:           {consecutive_rejections} / {MAX_REJECTIONS}")
print(f"sentinel checkpoint:              {SENTINEL_CHECKPOINT}")

stall_boost_at = max(1, math.ceil(MAX_REJECTIONS / 2))
if consecutive_rejections >= MAX_REJECTIONS:
    print("  -> at the forced-promotion threshold: the next rejection only forces a promotion")
    print(f"     through if the WORST LCB across every opponent clears {MIN_FORCE_PROMOTE_LCB:.0%} --")
    print("     the SAME statistical bar normal promotion uses (see --min-force-promote-lcb).")
elif consecutive_rejections >= stall_boost_at:
    print(f"  -> past the stall-boost threshold ({stall_boost_at}): evaluations are being")
    print("     widened automatically to cut through noise before any forced-promotion check")
print()

In [ ]:
# ---- Tune these per session; nothing below this block is hardcoded ----
ITERATIONS               = 100      # pipeline iterations to run this session
CONCURRENT_GAMES         = 350
GAMES_PER_ITERATION      = 500
MCTS_SIMS                = 500      # deeper self-play search
EVAL_GAMES               = 150      # deeper gating evaluation
EVAL_SIMS                = 300      # deeper gating search
BATCH_SIZE               = 2048
NUM_RES_BLOCKS           = 6
NUM_CHANNELS             = 128
MAX_REJECTIONS_ARG       = 5
MIN_FORCE_PROMOTE_LCB_ARG = 0.55    # stricter than the code's own 0.50 floor, on purpose
STALL_EVAL_MULTIPLIER    = 3
MAX_BUFFER_SIZE          = 1_000_000
BUFFER_ARCHIVE_INTERVAL  = 5

# Periodic drift check vs the pinned sentinel + recent milestones -- "LCB every
# few iterations" independent of the routine per-iteration gate above.
MILESTONE_INTERVAL       = 25
MILESTONE_GAMES          = 200
MILESTONE_SIMS           = 300
MILESTONE_MAX_REFS       = 5

args = [
    f"--iterations {ITERATIONS}",
    f"--concurrent-games {CONCURRENT_GAMES}",
    f"--games-per-iteration {GAMES_PER_ITERATION}",
    f"--mcts-sims {MCTS_SIMS}",
    f"--eval-games {EVAL_GAMES}",
    f"--eval-sims {EVAL_SIMS}",
    f"--batch-size {BATCH_SIZE}",
    f"--num-res-blocks {NUM_RES_BLOCKS}",
    f"--num-channels {NUM_CHANNELS}",
    f"--max-rejections {MAX_REJECTIONS_ARG}",
    f"--min-force-promote-lcb {MIN_FORCE_PROMOTE_LCB_ARG}",
    f"--stall-eval-multiplier {STALL_EVAL_MULTIPLIER}",
    f"--max-buffer-size {MAX_BUFFER_SIZE}",
    f"--buffer-archive-interval {BUFFER_ARCHIVE_INTERVAL}",
    f"--milestone-interval {MILESTONE_INTERVAL}",
    f"--milestone-games {MILESTONE_GAMES}",
    f"--milestone-sims {MILESTONE_SIMS}",
    f"--milestone-max-refs {MILESTONE_MAX_REFS}",
]
if SENTINEL_CHECKPOINT:
    args.append(f"--sentinel-checkpoint {SENTINEL_CHECKPOINT}")

cmd = "python python/pipeline.py " + " ".join(args)
print(cmd)
!{cmd}

In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/models/pipeline_state.json")), indent=2))

In [ ]:
import sys, glob
sys.path.append("/kaggle/working/zerocross-ai/python")
from plot_metrics import plot_training_metrics, plot_milestone_metrics
from IPython.display import Image, display

plot_training_metrics()
plot_milestone_metrics()
for img_path in sorted(glob.glob("/kaggle/working/models/plots/*.png")):
    display(Image(filename=img_path))

In [ ]:
# Optional: run this whenever you want a manual "is the champion actually still
# ahead of X" check outside the training loop -- e.g. after a milestone-gauntlet
# drift warning above, or just to compare against a batch of old rollback
# candidates. Purely informational -- never touches training/promotion state.
RUN_ARENA = False

if RUN_ARENA:
    import glob

    checkpoint_args = ["--checkpoint", "models/best_model.pth"]
    if SENTINEL_CHECKPOINT:
        checkpoint_args += ["--checkpoint", SENTINEL_CHECKPOINT]

    # Sample the most recent historical champions too, if any exist
    history = sorted(
        glob.glob("models/champion_gen_*.pth"),
        key=lambda p: int(''.join(filter(str.isdigit, p.split('_')[-1]))) if any(c.isdigit() for c in p) else -1
    )[-3:]
    for h in history:
        checkpoint_args += ["--checkpoint", h]

    cmd = "python python/arena.py " + " ".join(checkpoint_args) + " --games 100 --sims 250"
    print(cmd)
    !{cmd}
else:
    print("RUN_ARENA is False -- set it to True above and re-run this cell to compare checkpoints manually.")